In [0]:
select *
From workspace.default.car_sales_dataset;


--Different year models from dataset--
select distinct year
from workspace.default.car_sales_dataset
order by year desc;


--Distinct car makes from dataset--
select distinct make
from workspace.default.car_sales_dataset
order by make ASC;


--Different body types from dataset--
select distinct body
from workspace.default.car_sales_dataset
order by body ASC;


--Different transmission types from dataset--
select distinct transmission   
from workspace.default.car_sales_dataset
order by transmission ASC;


--States selling the vehicles--
select distinct state
from workspace.default.car_sales_dataset
order by state ASC;


--Different colors of vehicles--
select distinct color
from workspace.default.car_sales_dataset;


--Different interiors of vehicles--
select distinct interior
from workspace.default.car_sales_dataset;


-- Revenue per Month and per location
SELECT DISTINCT
  MONTHNAME(TRY_TO_TIMESTAMP(SUBSTRING(saledate, 5), 'MMM dd yyyy HH:mm:ss')) AS Month_name,
  COUNT(DISTINCT vin) AS number_of_sales,
  SUM(sellingprice) AS revenue_per_month,
  state
FROM
  workspace.default.car_sales_dataset
GROUP BY
  MONTHNAME(TRY_TO_TIMESTAMP(SUBSTRING(saledate, 5), 'MMM dd yyyy HH:mm:ss')),
  state;

--Final analysis code --

WITH base AS (
    SELECT
        vin,
        year,
        make,
        body,
        transmission,
        state,
        color,
        interior,
        sellingprice,

        -- Parse ONCE only, tolerate invalid input
        TRY_TO_TIMESTAMP(SUBSTRING(saledate, 5), 'MMM dd yyyy HH:mm:ss') AS sale_ts
    FROM workspace.default.car_sales_dataset
    WHERE saledate IS NOT NULL
),

enriched AS (
    SELECT
        vin,
        year,
        make,
        body,
        transmission,
        state,
        color,
        interior,
        sellingprice,

        YEAR(sale_ts) AS sale_year,
        MONTH(sale_ts) AS sale_month,
        DATE_FORMAT(sale_ts, 'MMMM') AS sale_month_name
    FROM base
    WHERE sale_ts IS NOT NULL
)

SELECT
    sale_year,
    sale_month,
    sale_month_name,
    state,
    make,
    body,
    transmission,

    COUNT(*) AS total_sales,  -- faster than COUNT DISTINCT vin if vin is unique
    SUM(sellingprice) AS total_revenue,
    ROUND(AVG(sellingprice), 2) AS avg_price,
    MIN(sellingprice) AS min_price,
    MAX(sellingprice) AS max_price,

    -- Use approx for large datasets (faster)
    APPROX_COUNT_DISTINCT(color) AS unique_colors_sold,
    APPROX_COUNT_DISTINCT(interior) AS unique_interiors_sold,

    ROUND(SUM(sellingprice) / COUNT(*), 2) AS revenue_per_sale,

    -- Window functions (optimized with partitioning)
    RANK() OVER (
        PARTITION BY sale_year, sale_month, state
        ORDER BY SUM(sellingprice) DESC
    ) AS revenue_rank_in_state,

    RANK() OVER (
        PARTITION BY sale_year, sale_month
        ORDER BY COUNT(*) DESC
    ) AS popularity_rank

FROM enriched
GROUP BY
    sale_year,
    sale_month,
    sale_month_name,
    state,
    make,
    body,
    transmission
ORDER BY
    sale_year DESC,
    sale_month DESC,
    total_revenue DESC;